Day 3 — Feature engineering: speed, fare per mile, tip %, time buckets, group by insights, iterative cleaning

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/cleaned_data.csv')

In [4]:
df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,trip_duration_mins,pickup_hour,pickup_day,pickup_month
0,2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,N,-73.974785,...,1.0,0.5,3.25,0.0,0.3,17.05,18.05,19,Thursday,1
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.001648,40.724243,1,N,-73.994415,...,0.5,0.5,2.00,0.0,0.3,17.80,19.83,20,Saturday,1
2,1,2015-01-10 20:33:38,2015-01-10 20:43:41,1,1.80,-73.963341,40.802788,1,N,-73.951820,...,0.5,0.5,0.00,0.0,0.3,10.80,10.05,20,Saturday,1
3,1,2015-01-10 20:33:39,2015-01-10 20:35:31,1,0.50,-74.009087,40.713818,1,N,-74.004326,...,0.5,0.5,0.00,0.0,0.3,4.80,1.87,20,Saturday,1
4,1,2015-01-10 20:33:39,2015-01-10 20:52:58,1,3.00,-73.971176,40.762428,1,N,-74.004181,...,0.5,0.5,0.00,0.0,0.3,16.30,19.32,20,Saturday,1


Speed

In [22]:
df['speed_mph'] = df['trip_distance']/(df['trip_duration_mins']/60)
df.sort_values(by='speed_mph', ascending=False).head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,improvement_surcharge,total_amount,trip_duration_mins,pickup_hour,pickup_day,pickup_month,speed_mph,fare_per_mile,time_of_day,tip_pct
24395,1,2015-01-19 15:11:32,2015-01-19 15:11:32,1,10.1,-73.972870,40.755802,1,N,-73.972870,...,0.3,34.63,0.00,15,Monday,1,inf,2.821782,off_peak,0.0
87524,1,2015-01-22 02:59:43,2015-01-22 02:59:43,1,11.2,-73.836777,40.686852,1,Y,-73.836777,...,0.3,33.80,0.00,2,Thursday,1,inf,2.901786,late_night,0.0
76947,1,2015-01-07 11:56:56,2015-01-07 11:56:56,1,1.0,-73.996536,40.763084,1,N,-73.996536,...,0.3,3.30,0.00,11,Wednesday,1,inf,2.500000,off_peak,0.0
86635,2,2015-01-20 19:49:16,2015-01-20 19:49:17,1,16.9,0.000000,0.000000,2,N,0.000000,...,0.3,63.20,0.02,19,Tuesday,1,50700.0,3.076923,evening_rush,20.0
49648,1,2015-01-23 10:18:22,2015-01-23 10:18:25,1,17.4,-73.983421,40.689159,1,N,-73.983421,...,0.3,3.30,0.05,10,Friday,1,20880.0,0.143678,off_peak,0.0


Fare Per Mile

In [21]:
import numpy as np

df['fare_per_mile'] = np.where(
    df['trip_distance'] == 0,
    np.nan,
    df['fare_amount'] / df['trip_distance']
)
df.sort_values(by='fare_per_mile', ascending=False).head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,improvement_surcharge,total_amount,trip_duration_mins,pickup_hour,pickup_day,pickup_month,speed_mph,fare_per_mile,time_of_day,tip_pct
93180,2,2015-01-23 23:39:05,2015-01-23 23:42:05,1,0.02,-73.751183,41.002159,5,N,-73.750900,...,0.3,139.7,3.00,23,Friday,1,0.400000,6945.0,off_peak,0.00
75645,2,2015-01-19 16:34:36,2015-01-19 16:35:16,6,0.01,-73.776390,40.645432,5,N,-73.776337,...,0.3,74.7,0.67,16,Monday,1,0.895522,6200.0,off_peak,20.00
31,2,2015-01-15 19:05:43,2015-01-15 19:05:44,2,0.01,0.000000,0.000000,5,N,0.000000,...,0.3,60.3,0.02,19,Thursday,1,30.000000,6000.0,evening_rush,0.00
36007,2,2015-01-14 04:40:05,2015-01-14 04:40:20,2,0.01,-73.790100,40.646870,2,N,-73.790283,...,0.3,69.6,0.25,4,Wednesday,1,2.400000,5200.0,late_night,22.06
12891,2,2015-01-12 13:11:08,2015-01-12 13:11:15,3,0.01,-73.870705,40.773640,2,N,-73.871651,...,0.3,52.8,0.12,13,Monday,1,5.000000,5200.0,off_peak,0.00


Time Of day Buckets

In [8]:
def time_bucket(hour):
  if hour in range(7,10):
    return  'morning_rush'
  elif hour in range(17,20):
    return 'evening_rush'
  elif hour in range(0,6):
    return 'late_night'
  else:
    return 'off_peak'

df['time_of_day'] = df['pickup_hour'].apply(time_bucket)
df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,tolls_amount,improvement_surcharge,total_amount,trip_duration_mins,pickup_hour,pickup_day,pickup_month,speed_mph,fare_per_mile,time_of_day
0,2,2015-01-15 19:05:39,2015-01-15 19:23:42,1,1.59,-73.993896,40.750111,1,N,-73.974785,...,0.0,0.3,17.05,18.05,19,Thursday,1,5.285319,7.547170,evening_rush
1,1,2015-01-10 20:33:38,2015-01-10 20:53:28,1,3.30,-74.001648,40.724243,1,N,-73.994415,...,0.0,0.3,17.80,19.83,20,Saturday,1,9.984871,4.393939,off_peak
2,1,2015-01-10 20:33:38,2015-01-10 20:43:41,1,1.80,-73.963341,40.802788,1,N,-73.951820,...,0.0,0.3,10.80,10.05,20,Saturday,1,10.746269,5.277778,off_peak
3,1,2015-01-10 20:33:39,2015-01-10 20:35:31,1,0.50,-74.009087,40.713818,1,N,-74.004326,...,0.0,0.3,4.80,1.87,20,Saturday,1,16.042781,7.000000,off_peak
4,1,2015-01-10 20:33:39,2015-01-10 20:52:58,1,3.00,-73.971176,40.762428,1,N,-74.004181,...,0.0,0.3,16.30,19.32,20,Saturday,1,9.316770,5.000000,off_peak


Tip Percentage

In [24]:
df['tip_pct'] = np.where(
    df['fare_amount'] == 0,
    np.nan,
    df['tip_amount']/df['fare_amount']
).round(2)
df.sort_values(by='tip_amount', ascending=False)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RateCodeID,store_and_fwd_flag,dropoff_longitude,...,improvement_surcharge,total_amount,trip_duration_mins,pickup_hour,pickup_day,pickup_month,speed_mph,fare_per_mile,time_of_day,tip_pct
31151,2,2015-01-29 02:40:59,2015-01-29 02:41:00,2,0.00,0.000000,0.000000,1,N,0.000000,...,0.3,103.80,0.02,2,Thursday,1,0.000000,NaN,late_night,40.00
41790,1,2015-01-25 07:22:54,2015-01-25 07:26:12,1,0.80,-73.988617,40.778526,1,N,-73.976311,...,0.3,105.30,3.30,7,Sunday,1,14.545455,5.625000,morning_rush,22.22
50272,1,2015-01-21 07:10:32,2015-01-21 07:10:47,1,0.00,-73.980904,40.763569,1,N,-73.981873,...,0.3,90.00,0.25,7,Wednesday,1,0.000000,NaN,morning_rush,34.68
72920,2,2015-01-13 08:33:07,2015-01-13 09:21:33,1,24.97,-73.997681,40.756660,1,N,-74.361740,...,0.3,147.90,48.43,8,Tuesday,1,30.935371,2.703244,morning_rush,0.98
66479,2,2015-01-22 22:30:11,2015-01-22 22:34:41,2,1.13,-73.994705,40.745831,1,N,-74.000389,...,0.3,67.80,4.50,22,Thursday,1,15.066667,4.867257,off_peak,11.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22473,2,2015-01-07 16:14:36,2015-01-07 16:26:16,1,1.31,-73.976456,40.788303,1,N,-73.958069,...,0.3,10.80,11.67,16,Wednesday,1,6.735219,6.870229,off_peak,0.00
4924,2,2015-01-15 17:33:24,2015-01-15 17:33:31,2,0.00,-73.982567,40.739799,1,N,-73.982567,...,0.3,-5.00,0.12,17,Thursday,1,0.000000,NaN,evening_rush,0.28
19953,2,2015-01-10 02:23:53,2015-01-10 02:23:58,2,0.00,0.000000,0.000000,5,N,0.000000,...,0.3,-8.10,0.08,2,Saturday,1,0.000000,NaN,late_night,0.15
22989,2,2015-01-14 11:52:09,2015-01-14 11:52:20,1,0.00,-73.789955,40.646946,2,N,0.000000,...,0.3,-72.46,0.18,11,Wednesday,1,0.000000,NaN,off_peak,0.28


In [25]:
print(df[['speed_mph', 'fare_per_mile', 'tip_pct']].describe())

          speed_mph  fare_per_mile       tip_pct
count  9.989800e+04   99366.000000  99962.000000
mean            inf       6.280120      0.130710
std             NaN      54.791693      0.234215
min    0.000000e+00   -6945.000000      0.000000
25%    8.311688e+00       4.166667      0.000000
50%    1.095816e+01       5.228758      0.150000
75%    1.460870e+01       6.551724      0.220000
max             inf    6945.000000     40.000000


In [26]:
# Removing rows that cause infinity in speed
df = df[df['trip_duration_mins'] > 0.5]
df = df[df['trip_distance'] > 0.1]

In [27]:
# Recalculating features
df['speed_mph'] = df['trip_distance'] / (df['trip_duration_mins'] / 60)
df['fare_per_mile'] = df['fare_amount'] / df['trip_distance']


In [29]:
# Removing impossible speeds (over 100mph in NYC is impossible)
df = df[df['speed_mph'] < 100]
df = df[df['fare_per_mile'] > 0]

print(f"Rows remaining: {len(df)}")
print(df[['speed_mph','fare_per_mile','tip_pct']].describe())

Rows remaining: 99061
          speed_mph  fare_per_mile       tip_pct
count  99061.000000   99061.000000  99061.000000
mean      12.402195       5.655724      0.129746
std        6.234586       3.010409      0.146045
min        0.020891       0.002500      0.000000
25%        8.367347       4.166667      0.000000
50%       10.992835       5.227273      0.150000
75%       14.634146       6.538462      0.220000
max       97.807333     300.000000     22.220000


In [30]:
print(df.groupby('time_of_day')['fare_per_mile'].mean().round(2))
print(df.groupby('pickup_day')['tip_pct'].mean().round(2))
print(df.groupby('passenger_count')['tip_pct'].mean().round(2))

time_of_day
evening_rush    5.75
late_night      4.91
morning_rush    5.91
off_peak        5.71
Name: fare_per_mile, dtype: float64
pickup_day
Friday       0.13
Monday       0.13
Saturday     0.12
Sunday       0.13
Thursday     0.13
Tuesday      0.13
Wednesday    0.13
Name: tip_pct, dtype: float64
passenger_count
0    0.16
1    0.13
2    0.13
3    0.12
4    0.11
5    0.13
6    0.13
Name: tip_pct, dtype: float64
